# Tire System Calculator

## Goal

Build and validate a Python calculator that ranks road tire pairings using
pressure, rolling resistance, mounted width, road surface and aerodynamic
performance. The notebook also packages the model as a static Gradio Lite app
for GitHub Pages.

## Setup

The notebook is the canonical source for the calculation model and the static
app. All app-runtime cells are tagged `gradio-lite` so the final build step can
package the same Python source without a separate script.

In [1]:
import base64
import hashlib
import html
import io
import json
import math
import urllib.request
import zipfile
from pathlib import Path

import pandas as pd

In [2]:
gradio_runtime_imports = """import html
import json
import math
from pathlib import Path

import gradio as gr
import pandas as pd"""

## Steps

### 1. Define tire test inputs

In [3]:
tire_specs = pd.DataFrame(
    [
        {
            "tire_id": "str25",
            "name": "Continental GP5000 S TR 25",
            "short_name": "S TR 25",
            "family": "str",
            "nominal_width_mm": 25,
            "base_watts": 10.1,
            "reference_pressure_psi": 80,
            "aero_watts_at_40_kmh": -0.6,
            "width_model": "linear",
            "measured_width_mm": 25.3,
            "reference_internal_width_mm": 17.8,
            "front_only": False,
            "confidence": "high",
        },
        {
            "tire_id": "str28",
            "name": "Continental GP5000 S TR 28",
            "short_name": "S TR 28",
            "family": "str",
            "nominal_width_mm": 28,
            "base_watts": 9.7,
            "reference_pressure_psi": 72,
            "aero_watts_at_40_kmh": 0.0,
            "width_model": "linear",
            "measured_width_mm": 28.0,
            "reference_internal_width_mm": 17.8,
            "front_only": False,
            "confidence": "high",
        },
        {
            "tire_id": "str30",
            "name": "Continental GP5000 S TR 30",
            "short_name": "S TR 30",
            "family": "str",
            "nominal_width_mm": 30,
            "base_watts": 10.0,
            "reference_pressure_psi": 69,
            "aero_watts_at_40_kmh": 1.1,
            "width_model": "linear",
            "measured_width_mm": 29.8,
            "reference_internal_width_mm": 17.8,
            "front_only": False,
            "confidence": "high",
        },
        {
            "tire_id": "str32",
            "name": "Continental GP5000 S TR 32",
            "short_name": "S TR 32",
            "family": "str",
            "nominal_width_mm": 32,
            "base_watts": 9.8,
            "reference_pressure_psi": 64,
            "aero_watts_at_40_kmh": 2.55,
            "width_model": "linear",
            "measured_width_mm": 31.4,
            "reference_internal_width_mm": 17.8,
            "front_only": False,
            "confidence": "medium",
        },
        {
            "tire_id": "str35",
            "name": "Continental GP5000 S TR 35",
            "short_name": "S TR 35",
            "family": "str",
            "nominal_width_mm": 35,
            "base_watts": 9.6,
            "reference_pressure_psi": 58,
            "aero_watts_at_40_kmh": 6.45,
            "width_model": "linear",
            "measured_width_mm": 34.0,
            "reference_internal_width_mm": 17.8,
            "front_only": False,
            "confidence": "medium",
        },
        {
            "tire_id": "aero26",
            "name": "Continental AERO 111 26",
            "short_name": "AERO 111 26",
            "family": "aero111",
            "nominal_width_mm": 26,
            "base_watts": 10.5,
            "reference_pressure_psi": 80,
            "aero_watts_at_40_kmh": -0.29,
            "width_model": "linear",
            "measured_width_mm": 25.68,
            "reference_internal_width_mm": 22.0,
            "front_only": True,
            "confidence": "high",
        },
        {
            "tire_id": "aero29",
            "name": "Continental AERO 111 29",
            "short_name": "AERO 111 29",
            "family": "aero111",
            "nominal_width_mm": 29,
            "base_watts": 10.5,
            "reference_pressure_psi": 72,
            "aero_watts_at_40_kmh": -1.23,
            "width_model": "linear",
            "measured_width_mm": 28.81,
            "reference_internal_width_mm": 22.0,
            "front_only": True,
            "confidence": "high",
        },
        {
            "tire_id": "slr28",
            "name": "Pirelli P Zero Race TLR SL-R 28",
            "short_name": "SL-R 28",
            "family": "slr",
            "nominal_width_mm": 28,
            "base_watts": 8.4,
            "reference_pressure_psi": 72,
            "aero_watts_at_40_kmh": -1.05,
            "width_model": "slr28_box",
            "measured_width_mm": 28.5,
            "reference_internal_width_mm": 19.0,
            "front_only": False,
            "confidence": "high",
        },
        {
            "tire_id": "slr30",
            "name": "Pirelli P Zero Race TLR SL-R 30",
            "short_name": "SL-R 30",
            "family": "slr",
            "nominal_width_mm": 30,
            "base_watts": 8.5,
            "reference_pressure_psi": 67,
            "aero_watts_at_40_kmh": -0.6,
            "width_model": "linear",
            "measured_width_mm": 31.3,
            "reference_internal_width_mm": 23.5,
            "front_only": False,
            "confidence": "medium",
        },
        {
            "tire_id": "tt28",
            "name": "Continental GP5000 TT TR 28",
            "short_name": "TT TR 28",
            "family": "tt",
            "nominal_width_mm": 28,
            "base_watts": 8.3,
            "reference_pressure_psi": 72,
            "aero_watts_at_40_kmh": -0.3,
            "width_model": "linear",
            "measured_width_mm": 28.0,
            "reference_internal_width_mm": 19.0,
            "front_only": False,
            "confidence": "medium",
        },
        {
            "tire_id": "tt30",
            "name": "Continental GP5000 TT TR 30",
            "short_name": "TT TR 30",
            "family": "tt",
            "nominal_width_mm": 30,
            "base_watts": 8.5,
            "reference_pressure_psi": 67,
            "aero_watts_at_40_kmh": -0.1,
            "width_model": "linear",
            "measured_width_mm": 31.3,
            "reference_internal_width_mm": 23.5,
            "front_only": False,
            "confidence": "high",
        },
    ]
)

In [4]:
tire_specs.head()

,tire_id,name,short_name,family,nominal_width_mm,base_watts,reference_pressure_psi,aero_watts_at_40_kmh,width_model,measured_width_mm,reference_internal_width_mm,front_only,confidence
0,str25,Continental GP5000 S TR 25,S TR 25,str,25,10.1,80,-0.60,linear,25.3,17.8,False,high
1,str28,Continental GP5000 S TR 28,S TR 28,str,28,9.7,72,0.00,linear,28.0,17.8,False,high
2,str30,Continental GP5000 S TR 30,S TR 30,str,30,10.0,69,1.10,linear,29.8,17.8,False,high
3,str32,Continental GP5000 S TR 32,S TR 32,str,32,9.8,64,2.55,linear,31.4,17.8,False,medium
4,str35,Continental GP5000 S TR 35,S TR 35,str,35,9.6,58,6.45,linear,34.0,17.8,False,medium


### 2. Define model assumptions

In [5]:
gravity = 9.80665
brr_load_kg = 42.5
brr_speed_mps = 29 / 3.6

surface_specs = {
    "New pavement": {"k1": 261.0, "roughness": 0.006},
    "Worn pavement": {"k1": 246.5, "roughness": 0.022},
    "Poor pavement": {"k1": 225.0, "roughness": 0.07},
    "Cobbles": {"k1": 199.0, "roughness": 0.5},
}

bike_specs = {
    "TT / triathlon": {
        "front_load_fraction": 0.5,
        "front_pressure_coefficient": 1.0,
        "rear_pressure_coefficient": 1.0,
    },
    "Road race": {
        "front_load_fraction": 0.45,
        "front_pressure_coefficient": 0.985,
        "rear_pressure_coefficient": 1.01,
    },
    "Endurance": {
        "front_load_fraction": 0.44,
        "front_pressure_coefficient": 0.975,
        "rear_pressure_coefficient": 1.02,
    },
}

### 3. Calculate pressure, width and power

In [6]:
def clamp(value, lower_bound, upper_bound):
    return min(upper_bound, max(lower_bound, value))


def mounted_width_mm(tire, internal_width_mm):
    if tire["width_model"] == "slr28_box":
        if internal_width_mm <= 21:
            return 28.5 + 0.25 * (internal_width_mm - 19)
        return 29.0 + 0.5 * (internal_width_mm - 21)

    return tire["measured_width_mm"] + 0.4 * (
        internal_width_mm - tire["reference_internal_width_mm"]
    )


def silca_pressure_psi(
    system_weight_kg,
    mounted_width,
    speed_kmh,
    surface_name,
    bike_name,
    axle,
):
    pressure_factor = (
        0.5 * (system_weight_kg - 50)
        + surface_specs[surface_name]["k1"]
    )
    tire_radius_term = mounted_width + 622 / 2
    pressure_numerator = (
        -0.00006 * mounted_width**3
        + 0.0079 * mounted_width**2
        - 0.4102 * mounted_width
        + 12.725
    ) * -226.44
    pressure_denominator_term = (
        (-0.5 * 9.81)
        / (pressure_factor * (20 / mounted_width))
        + tire_radius_term
    )
    contact_patch_pressure = pressure_numerator / (
        pressure_denominator_term**2 - tire_radius_term**2
    )
    speed_mph = speed_kmh / 1.609344
    speed_coefficient = 0.97 + ((speed_mph - 10) / 23) * 0.06
    pressure_coefficient = bike_specs[bike_name][
        f"{axle}_pressure_coefficient"
    ]

    return clamp(
        contact_patch_pressure
        * speed_coefficient
        * pressure_coefficient,
        28,
        120,
    )


def rolling_watts(tire, pressure_psi, wheel_load_kg, speed_kmh):
    reference_crr = tire["base_watts"] / (
        brr_load_kg * gravity * brr_speed_mps
    )
    pressure_adjustment = (
        tire["reference_pressure_psi"] / pressure_psi
    ) ** 0.12

    return (
        reference_crr
        * pressure_adjustment
        * wheel_load_kg
        * gravity
        * (speed_kmh / 3.6)
    )


def surface_watts(
    mounted_width,
    wheel_load_kg,
    speed_kmh,
    surface_name,
):
    return (
        surface_specs[surface_name]["roughness"]
        * wheel_load_kg
        * (speed_kmh / 40) ** 2.2
        * (30 / mounted_width) ** 3
    )


def aero_watts(
    tire,
    mounted_width,
    external_width_mm,
    rim_depth_mm,
    speed_kmh,
    axle,
    internal_width_mm,
):
    speed_scale = (speed_kmh / 40) ** 3
    depth_scale = clamp(rim_depth_mm / 60, 0.65, 1.15)
    tire_rim_overlap = max(0, mounted_width - external_width_mm)

    if tire["family"] == "aero111":
        fit_rate = 0.15
    elif (
        tire["family"] == "slr"
        and 22 <= internal_width_mm <= 25
    ):
        fit_rate = 0.1
    else:
        fit_rate = 0.35

    base_aero_watts = tire["aero_watts_at_40_kmh"] * (
        depth_scale
        if tire["aero_watts_at_40_kmh"] < 0
        else 1.0
    ) + tire_rim_overlap * fit_rate

    return (
        base_aero_watts
        * speed_scale
        * (1.0 if axle == "front" else 0.2)
    )


def rank_tires(
    system_weight_kg,
    bike_name,
    surface_name,
    speed_from_kmh,
    speed_to_kmh,
    front_internal_width_mm,
    front_external_width_mm,
    front_rim_depth_mm,
    rear_internal_width_mm,
    rear_external_width_mm,
    rear_rim_depth_mm,
):
    lower_speed_kmh = min(speed_from_kmh, speed_to_kmh)
    upper_speed_kmh = max(speed_from_kmh, speed_to_kmh)
    speed_samples_kmh = [
        lower_speed_kmh
        + (upper_speed_kmh - lower_speed_kmh)
        * sample_number
        / 4
        for sample_number in range(5)
    ]
    midpoint_speed_kmh = (
        lower_speed_kmh + upper_speed_kmh
    ) / 2
    front_load_kg = (
        system_weight_kg
        * bike_specs[bike_name]["front_load_fraction"]
    )
    rear_load_kg = system_weight_kg - front_load_kg
    tires = tire_specs.to_dict("records")
    rear_tires = [
        tire for tire in tires if not tire["front_only"]
    ]
    result_rows = []

    for front_tire in tires:
        for rear_tire in rear_tires:
            front_width_mm = mounted_width_mm(
                front_tire,
                front_internal_width_mm,
            )
            rear_width_mm = mounted_width_mm(
                rear_tire,
                rear_internal_width_mm,
            )
            front_pressure_psi = silca_pressure_psi(
                system_weight_kg,
                front_width_mm,
                midpoint_speed_kmh,
                surface_name,
                bike_name,
                "front",
            )
            rear_pressure_psi = silca_pressure_psi(
                system_weight_kg,
                rear_width_mm,
                midpoint_speed_kmh,
                surface_name,
                bike_name,
                "rear",
            )
            rolling_samples_watts = []
            surface_samples_watts = []
            aero_samples_watts = []

            for speed_kmh in speed_samples_kmh:
                sample_front_pressure_psi = silca_pressure_psi(
                    system_weight_kg,
                    front_width_mm,
                    speed_kmh,
                    surface_name,
                    bike_name,
                    "front",
                )
                sample_rear_pressure_psi = silca_pressure_psi(
                    system_weight_kg,
                    rear_width_mm,
                    speed_kmh,
                    surface_name,
                    bike_name,
                    "rear",
                )
                rolling_samples_watts.append(
                    rolling_watts(
                        front_tire,
                        sample_front_pressure_psi,
                        front_load_kg,
                        speed_kmh,
                    )
                    + rolling_watts(
                        rear_tire,
                        sample_rear_pressure_psi,
                        rear_load_kg,
                        speed_kmh,
                    )
                )
                surface_samples_watts.append(
                    surface_watts(
                        front_width_mm,
                        front_load_kg,
                        speed_kmh,
                        surface_name,
                    )
                    + surface_watts(
                        rear_width_mm,
                        rear_load_kg,
                        speed_kmh,
                        surface_name,
                    )
                )
                aero_samples_watts.append(
                    aero_watts(
                        front_tire,
                        front_width_mm,
                        front_external_width_mm,
                        front_rim_depth_mm,
                        speed_kmh,
                        "front",
                        front_internal_width_mm,
                    )
                    + aero_watts(
                        rear_tire,
                        rear_width_mm,
                        rear_external_width_mm,
                        rear_rim_depth_mm,
                        speed_kmh,
                        "rear",
                        rear_internal_width_mm,
                    )
                )

            rolling_loss_watts = sum(
                rolling_samples_watts
            ) / len(rolling_samples_watts)
            surface_loss_watts = sum(
                surface_samples_watts
            ) / len(surface_samples_watts)
            aero_loss_watts = sum(
                aero_samples_watts
            ) / len(aero_samples_watts)
            result_rows.append(
                {
                    "front_tire": front_tire["short_name"],
                    "rear_tire": rear_tire["short_name"],
                    "front_pressure_psi": front_pressure_psi,
                    "rear_pressure_psi": rear_pressure_psi,
                    "front_width_mm": front_width_mm,
                    "rear_width_mm": rear_width_mm,
                    "rolling_watts": rolling_loss_watts,
                    "surface_watts": surface_loss_watts,
                    "aero_watts": aero_loss_watts,
                    "total_watts": (
                        rolling_loss_watts
                        + surface_loss_watts
                        + aero_loss_watts
                    ),
                    "confidence": (
                        "high"
                        if (
                            front_tire["confidence"] == "high"
                            and rear_tire["confidence"] == "high"
                        )
                        else "medium"
                    ),
                }
            )

    ranked_tires = (
        pd.DataFrame(result_rows)
        .sort_values("total_watts")
        .reset_index(drop=True)
        .assign(
            rank=lambda dataframe: dataframe.index + 1,
            gap_watts=lambda dataframe: (
                dataframe["total_watts"]
                - dataframe["total_watts"].min()
            ),
        )
    )

    return ranked_tires

### 4. Check the default scenario

In [7]:
default_ranking = rank_tires(
    system_weight_kg=90,
    bike_name="TT / triathlon",
    surface_name="Worn pavement",
    speed_from_kmh=34,
    speed_to_kmh=44,
    front_internal_width_mm=22,
    front_external_width_mm=31.5,
    front_rim_depth_mm=60,
    rear_internal_width_mm=22,
    rear_external_width_mm=31.5,
    rear_rim_depth_mm=60,
)

In [8]:
default_ranking.head(10)

,front_tire,rear_tire,front_pressure_psi,rear_pressure_psi,front_width_mm,rear_width_mm,rolling_watts,surface_watts,aero_watts,total_watts,confidence,rank,gap_watts
0,SL-R 28,SL-R 28,73.507102,73.507102,29.5,29.5,23.861290,1.990940,-1.196636,24.655594,high,1,0.000000
1,SL-R 28,TT TR 28,73.507102,74.707700,29.5,29.2,23.696362,2.021938,-1.054179,24.664121,medium,2,0.008527
2,SL-R 28,SL-R 30,73.507102,69.000711,29.5,30.7,23.990712,1.878710,-1.111162,24.758261,medium,3,0.102667
3,SL-R 28,TT TR 30,73.507102,69.000711,29.5,30.7,23.990712,1.878710,-1.016191,24.853232,high,4,0.197638
4,SL-R 30,SL-R 28,69.000711,73.507102,30.7,29.5,23.990712,1.878710,-0.769266,25.100157,medium,5,0.444563
5,SL-R 30,TT TR 28,69.000711,74.707700,30.7,29.2,23.825784,1.909709,-0.626809,25.108684,medium,6,0.453090
6,SL-R 30,SL-R 30,69.000711,69.000711,30.7,30.7,24.120134,1.766481,-0.683792,25.202824,medium,7,0.547230
7,TT TR 28,SL-R 28,74.707700,73.507102,29.2,29.5,23.696362,2.021938,-0.484353,25.233948,medium,8,0.578354
8,TT TR 28,TT TR 28,74.707700,74.707700,29.2,29.2,23.531434,2.052937,-0.341896,25.242475,medium,9,0.586881
9,SL-R 30,TT TR 30,69.000711,69.000711,30.7,30.7,24.120134,1.766481,-0.588821,25.297795,medium,10,0.642201


### 5. Build the Gradio interface

In [9]:
app_css = '''
.gradio-container {
    max-width: 980px !important;
    margin: 0 auto !important;
}
.section-label {
    color: #66706d;
    font-size: 0.72rem;
    font-weight: 700;
    letter-spacing: 0.12em;
    text-transform: uppercase;
}
.winner-card {
    background: #111b22;
    color: #ffffff;
    padding: 1.4rem;
    margin: 0.5rem 0 1rem;
}
.winner-kicker {
    color: #b9dc35;
    font-size: 0.72rem;
    font-weight: 700;
    letter-spacing: 0.1em;
    text-transform: uppercase;
}
.winner-grid {
    display: grid;
    grid-template-columns: 1fr 1fr;
    gap: 1rem;
    margin-top: 1rem;
}
.winner-grid h3 {
    font-size: 1.05rem;
    margin: 0.25rem 0 0.5rem;
}
.winner-grid p {
    color: #d4dcda;
    margin: 0;
}
.winner-total {
    border-top: 1px solid #445158;
    margin-top: 1rem;
    padding-top: 1rem;
}
.winner-total strong {
    color: #b9dc35;
    font-size: 1.65rem;
}
.model-note {
    color: #66706d;
    font-size: 0.82rem;
    line-height: 1.5;
}
@media (max-width: 640px) {
    .winner-grid {
        grid-template-columns: 1fr;
    }
}
'''


def format_outputs(ranked_tires):
    winner = ranked_tires.iloc[0]
    winner_html = f'''
    <section class="winner-card">
        <div class="winner-kicker">Fastest modeled system</div>
        <div class="winner-grid">
            <div>
                <small>FRONT</small>
                <h3>{winner["front_tire"]}</h3>
                <p>
                    {winner["front_pressure_psi"]:.1f} psi ·
                    {winner["front_width_mm"]:.1f} mm mounted
                </p>
            </div>
            <div>
                <small>REAR</small>
                <h3>{winner["rear_tire"]}</h3>
                <p>
                    {winner["rear_pressure_psi"]:.1f} psi ·
                    {winner["rear_width_mm"]:.1f} mm mounted
                </p>
            </div>
        </div>
        <div class="winner-total">
            <strong>{winner["total_watts"]:.1f} W</strong>
            <span> modeled tire-system loss</span>
        </div>
    </section>
    '''
    ranking_table = (
        ranked_tires
        .head(10)
        .loc[
            :,
            [
                "rank",
                "front_tire",
                "rear_tire",
                "front_pressure_psi",
                "rear_pressure_psi",
                "front_width_mm",
                "rear_width_mm",
                "total_watts",
                "gap_watts",
                "confidence",
            ],
        ]
        .rename(
            columns={
                "rank": "Rank",
                "front_tire": "Front",
                "rear_tire": "Rear",
                "front_pressure_psi": "Front psi",
                "rear_pressure_psi": "Rear psi",
                "front_width_mm": "Front mm",
                "rear_width_mm": "Rear mm",
                "total_watts": "Total W",
                "gap_watts": "Gap W",
                "confidence": "Confidence",
            }
        )
        .round(
            {
                "Front psi": 1,
                "Rear psi": 1,
                "Front mm": 1,
                "Rear mm": 1,
                "Total W": 1,
                "Gap W": 1,
            }
        )
    )

    return winner_html, ranking_table


default_winner_html, default_ranking_table = format_outputs(
    default_ranking
)

In [10]:
gradio_interface_source = r"""with gr.Blocks(
    title="Tire System Calculator",
    css=app_css,
) as demo:
    gr.Markdown(
        "### INPUTS",
        elem_classes=["section-label"],
    )

    with gr.Row():
        with gr.Column():
            system_weight = gr.Slider(
                minimum=55,
                maximum=150,
                value=90,
                step=1,
                label="Total system weight (kg)",
            )
            bike = gr.Dropdown(
                choices=list(bike_specs),
                value="TT / triathlon",
                label="Bike position",
            )
            surface = gr.Dropdown(
                choices=list(surface_specs),
                value="Worn pavement",
                label="Road surface",
            )
            with gr.Row():
                speed_from = gr.Slider(
                    minimum=20,
                    maximum=65,
                    value=34,
                    step=1,
                    label="Speed from (km/h)",
                )
                speed_to = gr.Slider(
                    minimum=20,
                    maximum=65,
                    value=44,
                    step=1,
                    label="Speed to (km/h)",
                )

        with gr.Column():
            gr.Markdown("**Front wheel**")
            with gr.Row():
                front_internal_width = gr.Number(
                    value=22,
                    label="Internal width (mm)",
                )
                front_external_width = gr.Number(
                    value=31.5,
                    label="External width (mm)",
                )
                front_rim_depth = gr.Number(
                    value=60,
                    label="Rim depth (mm)",
                )

            gr.Markdown("**Rear wheel**")
            with gr.Row():
                rear_internal_width = gr.Number(
                    value=22,
                    label="Internal width (mm)",
                )
                rear_external_width = gr.Number(
                    value=31.5,
                    label="External width (mm)",
                )
                rear_rim_depth = gr.Number(
                    value=60,
                    label="Rim depth (mm)",
                )

    calculate_button = gr.Button(
        "Calculate fastest system",
        variant="primary",
    )

    gr.Markdown(
        "### RECOMMENDATION",
        elem_classes=["section-label"],
    )
    winner_output = gr.HTML(value=default_winner_html)
    ranking_output = gr.Dataframe(
        value=default_ranking_table,
        interactive=False,
        label="Top 10 front and rear pairings",
    )
    gr.Markdown(
        '''
        <p class="model-note">
        Aero is a relative adjustment against a well-matched GP5000 S TR 28
        setup. Common bike and rider drag is excluded because it does not
        affect the ranking. Puncture risk is intentionally excluded.
        </p>
        '''
    )

    calculation_inputs = [
        system_weight,
        bike,
        surface,
        speed_from,
        speed_to,
        front_internal_width,
        front_external_width,
        front_rim_depth,
        rear_internal_width,
        rear_external_width,
        rear_rim_depth,
    ]

    calculate_button.click(
        fn=lambda *input_values: format_outputs(
            rank_tires(*input_values)
        ),
        inputs=calculation_inputs,
        outputs=[winner_output, ranking_output],
    )"""

### 6. Package the static Pages app

In [11]:
Path("../docs/assets/fixed").mkdir(parents=True, exist_ok=True)

gradio_wheel_url = (
    "https://cdn.jsdelivr.net/npm/@gradio/lite@5.45.0/dist/assets/"
    "gradio-5.45.0-cp312-none-any.whl"
)
huggingface_wheel_url = (
    "https://files.pythonhosted.org/packages/33/d5/"
    "d9e9b75d8dc9cf125fff16fb0cd51d864a29e8b46b6880d8808940989405/"
    "huggingface_hub-0.33.5-py3-none-any.whl"
)

with urllib.request.urlopen(gradio_wheel_url) as wheel_response:
    gradio_wheel_bytes = wheel_response.read()

with zipfile.ZipFile(io.BytesIO(gradio_wheel_bytes)) as source_wheel:
    wheel_files = {
        file_info.filename: source_wheel.read(file_info.filename)
        for file_info in source_wheel.infolist()
        if not file_info.is_dir()
    }

metadata_name = next(
    file_name
    for file_name in wheel_files
    if file_name.endswith(".dist-info/METADATA")
)
record_name = next(
    file_name
    for file_name in wheel_files
    if file_name.endswith(".dist-info/RECORD")
)
metadata_text = wheel_files[metadata_name].decode("utf-8").replace(
    "Requires-Dist: huggingface-hub<1.0,>=0.33.5",
    f"Requires-Dist: huggingface-hub @ {huggingface_wheel_url}",
)
wheel_files[metadata_name] = metadata_text.encode("utf-8")

record_rows = []
for file_name, file_bytes in sorted(wheel_files.items()):
    if file_name == record_name:
        continue
    file_digest = base64.urlsafe_b64encode(
        hashlib.sha256(file_bytes).digest()
    ).decode("ascii").rstrip("=")
    record_rows.append(
        f"{file_name},sha256={file_digest},{len(file_bytes)}"
    )
record_rows.append(f"{record_name},,")
wheel_files[record_name] = (
    "\n".join(record_rows) + "\n"
).encode("utf-8")

with zipfile.ZipFile(
    "../docs/assets/fixed/gradio-5.45.0-cp312-none-any.whl",
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as fixed_wheel:
    for file_name, file_bytes in sorted(wheel_files.items()):
        fixed_wheel.writestr(file_name, file_bytes)

with open("0_build.ipynb", encoding="utf-8") as notebook_file:
    saved_notebook = json.load(notebook_file)

gradio_model_source = "\n\n".join(
    "".join(cell["source"])
    for cell in saved_notebook["cells"]
    if (
        cell["cell_type"] == "code"
        and "gradio-lite" in cell.get("metadata", {}).get("tags", [])
    )
)
gradio_lite_python = (
    gradio_runtime_imports
    + "\n\n"
    + gradio_model_source
    + "\n\n"
    + gradio_interface_source
    + "\n\ndemo.launch()\n"
)

with open("../docs/index.html", "w", encoding="utf-8") as output_file:
    output_file.write(
        f'''<!doctype html>
<html lang="en">
<head>
    <meta charset="utf-8">
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <meta name="description" content="Compare road tires using pressure, rolling resistance, mounted width, road surface and aerodynamic performance.">
    <title>Tire System Calculator</title>
    <script>
        const NativeWorker = window.Worker;
        window.Worker = class extends NativeWorker {{
            postMessage(message, transfer) {{
                if (message?.type === "init-env") {{
                    const fixedMessage = structuredClone(message);
                    fixedMessage.data.gradioWheelUrl = new URL(
                        "./assets/fixed/gradio-5.45.0-cp312-none-any.whl",
                        window.location.href,
                    ).href;
                    return super.postMessage(fixedMessage, transfer);
                }}
                return super.postMessage(message, transfer);
            }}
        }};
    </script>
    <script type="module" crossorigin src="https://cdn.jsdelivr.net/npm/@gradio/lite@5.45.0/dist/lite.js"></script>
    <link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/@gradio/lite@5.45.0/dist/lite.css">
    <style>
        * {{ box-sizing: border-box; }}
        body {{
            margin: 0;
            background: #f4f6f2;
            color: #101515;
            font-family: system-ui, sans-serif;
        }}
        header {{
            background: #111b22;
            color: white;
            padding: 3rem max(1.25rem, calc((100vw - 980px) / 2));
        }}
        header small {{
            color: #b9dc35;
            font-weight: 700;
            letter-spacing: 0.12em;
        }}
        header h1 {{
            font-size: clamp(2.2rem, 6vw, 4.6rem);
            letter-spacing: -0.055em;
            line-height: 0.96;
            margin: 0.8rem 0 1rem;
            max-width: 800px;
        }}
        header p {{
            color: #c8d0cd;
            line-height: 1.6;
            margin: 0;
            max-width: 650px;
        }}
        .loading-note {{
            color: #66706d;
            font-size: 0.8rem;
            margin: 1rem auto;
            max-width: 980px;
            padding: 0 1rem;
        }}
    </style>
</head>
<body>
    <header>
        <small>PYTHON · GRADIO LITE · MODEL 01</small>
        <h1>Find the fastest tire for your system.</h1>
        <p>
            Pressure, rolling resistance, mounted width and aerodynamic
            behavior evaluated together.
        </p>
    </header>
    <p class="loading-note">
        The Python model runs in your browser. The first load typically takes
        10 seconds.
    </p>
    <gradio-lite theme="light">
        <gradio-file name="app.py" entrypoint>
{html.escape(gradio_lite_python)}
        </gradio-file>
    </gradio-lite>
</body>
</html>
'''
    )

### 7. Build the dependency-free Pyodide interface

GitHub Pages cannot run a Python server. This build keeps every model
calculation in Python while using a small browser bridge for the interface.
It requires only the Pyodide runtime and installs no Python packages.

In [12]:
with open("0_build.ipynb", encoding="utf-8") as notebook_file:
    saved_notebook = json.load(notebook_file)

assumptions_source = "".join(
    next(
        cell["source"]
        for cell in saved_notebook["cells"]
        if cell.get("id") == "a205fa95"
    )
)
calculation_source = "".join(
    next(
        cell["source"]
        for cell in saved_notebook["cells"]
        if cell.get("id") == "51b4adbe"
    )
)
pandas_ranking_source = '''    ranked_tires = (
        pd.DataFrame(result_rows)
        .sort_values("total_watts")
        .reset_index(drop=True)
        .assign(
            rank=lambda dataframe: dataframe.index + 1,
            gap_watts=lambda dataframe: (
                dataframe["total_watts"]
                - dataframe["total_watts"].min()
            ),
        )
    )

    return ranked_tires'''
python_ranking_source = '''    ranked_tires = sorted(
        result_rows,
        key=lambda result_row: result_row["total_watts"],
    )
    fastest_watts = ranked_tires[0]["total_watts"]
    for rank, result_row in enumerate(ranked_tires, start=1):
        result_row["rank"] = rank
        result_row["gap_watts"] = (
            result_row["total_watts"] - fastest_watts
        )

    return ranked_tires'''
browser_calculation_source = (
    calculation_source
    .replace(
        '    tires = tire_specs.to_dict("records")',
        "    tires = tire_specs",
    )
    .replace(pandas_ranking_source, python_ranking_source)
)
assert "pd." not in browser_calculation_source

tire_specs_json = json.dumps(
    tire_specs.to_dict("records"),
    separators=(",", ":"),
)
browser_model_source = (
    "import json\n\n"
    + f"tire_specs = json.loads({tire_specs_json!r})\n\n"
    + assumptions_source
    + "\n\n"
    + browser_calculation_source
    + '''


def calculate_json(input_json):
    calculation_inputs = json.loads(input_json)
    ranked_tires = rank_tires(**calculation_inputs)
    return json.dumps(ranked_tires[:10])
'''
)

standalone_html = r'''<!doctype html>
<html lang="en">
<head>
    <meta charset="utf-8">
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <meta name="description" content="Compare road tires using pressure, rolling resistance, mounted width, road surface and aerodynamic performance.">
    <title>Tire System Calculator</title>
    <script src="https://cdn.jsdelivr.net/pyodide/v0.27.7/full/pyodide.js"></script>
    <style>
        :root {
            color-scheme: light;
            font-family: Inter, ui-sans-serif, system-ui, sans-serif;
        }
        * { box-sizing: border-box; }
        body {
            margin: 0;
            background: #f4f6f2;
            color: #101515;
        }
        header {
            background: #111b22;
            color: #ffffff;
            padding: 3rem max(1.25rem, calc((100vw - 1100px) / 2));
        }
        header small, .section-label {
            color: #9bbd22;
            font-size: 0.75rem;
            font-weight: 800;
            letter-spacing: 0.12em;
            text-transform: uppercase;
        }
        header h1 {
            font-size: clamp(2.3rem, 6vw, 4.8rem);
            letter-spacing: -0.055em;
            line-height: 0.96;
            margin: 0.8rem 0 1rem;
            max-width: 820px;
        }
        header p {
            color: #c8d0cd;
            line-height: 1.6;
            margin: 0;
            max-width: 680px;
        }
        main {
            margin: 0 auto;
            max-width: 1100px;
            padding: 1.5rem 1rem 4rem;
        }
        #runtime-status {
            color: #66706d;
            font-size: 0.88rem;
            margin: 0 0 1rem;
        }
        #runtime-status[data-state="error"] { color: #a71919; }
        .panel {
            background: #ffffff;
            border: 1px solid #dce2de;
            border-radius: 12px;
            padding: 1.25rem;
        }
        .input-grid {
            display: grid;
            gap: 1.5rem;
            grid-template-columns: repeat(2, minmax(0, 1fr));
        }
        .field-grid {
            display: grid;
            gap: 0.8rem;
            grid-template-columns: repeat(2, minmax(0, 1fr));
        }
        .wheel-grid {
            display: grid;
            gap: 0.8rem;
            grid-template-columns: repeat(3, minmax(0, 1fr));
        }
        h2, h3 { margin-top: 0; }
        h3 { margin-bottom: 0.75rem; }
        label {
            color: #35413e;
            display: grid;
            font-size: 0.8rem;
            font-weight: 700;
            gap: 0.35rem;
        }
        input, select, button {
            border: 1px solid #bcc6c1;
            border-radius: 8px;
            font: inherit;
            min-height: 44px;
            padding: 0.65rem 0.75rem;
            width: 100%;
        }
        button {
            background: #111b22;
            border-color: #111b22;
            color: #ffffff;
            cursor: pointer;
            font-weight: 800;
            margin-top: 1rem;
        }
        button:disabled { cursor: wait; opacity: 0.55; }
        .results { margin-top: 1.5rem; }
        .winner-card {
            background: #111b22;
            border-radius: 12px;
            color: #ffffff;
            padding: 1.4rem;
        }
        .winner-grid {
            display: grid;
            gap: 1rem;
            grid-template-columns: repeat(2, minmax(0, 1fr));
            margin-top: 1rem;
        }
        .winner-card h3 { margin: 0.2rem 0 0.4rem; }
        .winner-card p { color: #d4dcda; margin: 0; }
        .winner-total {
            border-top: 1px solid #445158;
            margin-top: 1rem;
            padding-top: 1rem;
        }
        .winner-total strong { color: #b9dc35; font-size: 1.65rem; }
        .table-wrap {
            margin-top: 1rem;
            overflow-x: auto;
        }
        table {
            border-collapse: collapse;
            font-size: 0.8rem;
            width: 100%;
        }
        th, td {
            border-bottom: 1px solid #e1e6e3;
            padding: 0.65rem;
            text-align: right;
            white-space: nowrap;
        }
        th:nth-child(2), th:nth-child(3), td:nth-child(2), td:nth-child(3) {
            text-align: left;
        }
        .model-note {
            color: #66706d;
            font-size: 0.82rem;
            line-height: 1.5;
        }
        @media (max-width: 760px) {
            .input-grid, .winner-grid { grid-template-columns: 1fr; }
            .wheel-grid { grid-template-columns: 1fr; }
        }
    </style>
</head>
<body data-ready="false">
    <header>
        <small>PYTHON · PYODIDE · MODEL 01</small>
        <h1>Find the fastest tire for your system.</h1>
        <p>Pressure, rolling resistance, mounted width and aerodynamic behavior evaluated together.</p>
    </header>
    <main>
        <p id="runtime-status" data-state="loading">Loading the Python model…</p>
        <section class="panel">
            <div class="input-grid">
                <div>
                    <p class="section-label">Rider and conditions</p>
                    <div class="field-grid">
                        <label>Total system weight (lb)<input id="system-weight" type="number" min="120" max="330" step="1" value="198"></label>
                        <label>Bike position<select id="bike"><option>TT / triathlon</option><option>Road race</option><option>Endurance</option></select></label>
                        <label>Road surface<select id="surface"><option>New pavement</option><option selected>Worn pavement</option><option>Poor pavement</option><option>Cobbles</option></select></label>
                        <label>Speed from (mph)<input id="speed-from" type="number" min="12" max="40" step="1" value="21"></label>
                        <label>Speed to (mph)<input id="speed-to" type="number" min="12" max="40" step="1" value="27"></label>
                    </div>
                </div>
                <div>
                    <p class="section-label">Wheel dimensions</p>
                    <h3>Front wheel</h3>
                    <div class="wheel-grid">
                        <label>Internal width (mm)<input id="front-internal" type="number" step="0.1" value="22"></label>
                        <label>External width (mm)<input id="front-external" type="number" step="0.1" value="31.5"></label>
                        <label>Rim depth (mm)<input id="front-depth" type="number" step="1" value="60"></label>
                    </div>
                    <h3 style="margin-top: 1rem">Rear wheel</h3>
                    <div class="wheel-grid">
                        <label>Internal width (mm)<input id="rear-internal" type="number" step="0.1" value="22"></label>
                        <label>External width (mm)<input id="rear-external" type="number" step="0.1" value="31.5"></label>
                        <label>Rim depth (mm)<input id="rear-depth" type="number" step="1" value="60"></label>
                    </div>
                </div>
            </div>
            <button id="calculate-button" type="button" disabled>Loading Python…</button>
        </section>
        <section class="results" aria-live="polite">
            <p class="section-label">Recommendation</p>
            <div id="winner-output"></div>
            <div class="table-wrap"><table id="ranking-table"></table></div>
            <p class="model-note">Aero is a relative adjustment against a well-matched GP5000 S TR 28 setup. Common bike and rider drag is excluded because it does not affect the ranking. Puncture risk is intentionally excluded.</p>
        </section>
    </main>
    <script>
        const pythonModelSource = __PYTHON_MODEL_SOURCE__;
        const runtimeStatus = document.getElementById("runtime-status");
        const calculateButton = document.getElementById("calculate-button");
        let pythonRuntime;

        function numericValue(elementId) {
            return Number(document.getElementById(elementId).value);
        }

        function calculatorInputs() {
            return {
                system_weight_kg: numericValue("system-weight") * 0.45359237,
                bike_name: document.getElementById("bike").value,
                surface_name: document.getElementById("surface").value,
                speed_from_kmh: numericValue("speed-from") * 1.609344,
                speed_to_kmh: numericValue("speed-to") * 1.609344,
                front_internal_width_mm: numericValue("front-internal"),
                front_external_width_mm: numericValue("front-external"),
                front_rim_depth_mm: numericValue("front-depth"),
                rear_internal_width_mm: numericValue("rear-internal"),
                rear_external_width_mm: numericValue("rear-external"),
                rear_rim_depth_mm: numericValue("rear-depth")
            };
        }

        function fixedNumber(value) {
            return Number(value).toFixed(1);
        }

        function renderResults(results) {
            const winner = results[0];
            document.getElementById("winner-output").innerHTML = `
                <section class="winner-card" data-testid="winner-card">
                    <div class="section-label">Fastest modeled system</div>
                    <div class="winner-grid">
                        <div><small>FRONT</small><h3>${winner.front_tire}</h3><p>${fixedNumber(winner.front_pressure_psi)} psi · ${fixedNumber(winner.front_width_mm)} mm mounted</p></div>
                        <div><small>REAR</small><h3>${winner.rear_tire}</h3><p>${fixedNumber(winner.rear_pressure_psi)} psi · ${fixedNumber(winner.rear_width_mm)} mm mounted</p></div>
                    </div>
                    <div class="winner-total"><strong>${fixedNumber(winner.total_watts)} W</strong><span> modeled tire-system loss</span></div>
                </section>`;

            const headings = ["Rank", "Front", "Rear", "Front psi", "Rear psi", "Front mm", "Rear mm", "Total W", "Gap W", "Confidence"];
            const rows = results.map((result) => [
                result.rank,
                result.front_tire,
                result.rear_tire,
                fixedNumber(result.front_pressure_psi),
                fixedNumber(result.rear_pressure_psi),
                fixedNumber(result.front_width_mm),
                fixedNumber(result.rear_width_mm),
                fixedNumber(result.total_watts),
                fixedNumber(result.gap_watts),
                result.confidence
            ]);
            document.getElementById("ranking-table").innerHTML = `
                <thead><tr>${headings.map((heading) => `<th>${heading}</th>`).join("")}</tr></thead>
                <tbody>${rows.map((row) => `<tr>${row.map((value) => `<td>${value}</td>`).join("")}</tr>`).join("")}</tbody>`;
        }

        async function calculate() {
            calculateButton.disabled = true;
            calculateButton.textContent = "Calculating…";
            try {
                pythonRuntime.globals.set(
                    "calculator_input_json",
                    JSON.stringify(calculatorInputs())
                );
                const resultJson = pythonRuntime.runPython(
                    "calculate_json(calculator_input_json)"
                );
                renderResults(JSON.parse(resultJson));
                runtimeStatus.textContent = "Python model ready";
                runtimeStatus.dataset.state = "ready";
                document.body.dataset.ready = "true";
            } catch (error) {
                runtimeStatus.textContent = `Calculation failed: ${error.message}`;
                runtimeStatus.dataset.state = "error";
                document.body.dataset.ready = "error";
                throw error;
            } finally {
                calculateButton.disabled = false;
                calculateButton.textContent = "Calculate fastest system";
            }
        }

        async function initializeCalculator() {
            try {
                pythonRuntime = await loadPyodide();
                await pythonRuntime.runPythonAsync(pythonModelSource);
                calculateButton.addEventListener("click", calculate);
                await calculate();
            } catch (error) {
                runtimeStatus.textContent = `Model failed to load: ${error.message}`;
                runtimeStatus.dataset.state = "error";
                document.body.dataset.ready = "error";
                console.error(error);
            }
        }

        initializeCalculator();
    </script>
</body>
</html>
'''
standalone_html = standalone_html.replace(
    "__PYTHON_MODEL_SOURCE__",
    json.dumps(browser_model_source),
)

with open("../docs/index.html", "w", encoding="utf-8") as output_file:
    output_file.write(standalone_html)

Path(
    "../docs/assets/fixed/gradio-5.45.0-cp312-none-any.whl"
).unlink(missing_ok=True)

## Checks

In [13]:
assert len(tire_specs) == 11
assert default_ranking.shape[0] == 99
assert default_ranking["total_watts"].is_monotonic_increasing
assert Path("../docs/index.html").exists()
assert not Path(
    "../docs/assets/fixed/gradio-5.45.0-cp312-none-any.whl"
).exists()

with open("../docs/index.html", encoding="utf-8") as output_file:
    generated_html = output_file.read()

assert "pyodide/v0.27.7/full/pyodide.js" in generated_html
assert "data-testid=\"winner-card\"" in generated_html
assert "@gradio/lite" not in generated_html
assert "huggingface" not in generated_html.lower()
assert "Continental GP5000 S TR 35" in generated_html
assert "Pirelli P Zero Race TLR SL-R 30" in generated_html
assert "calculate_json(calculator_input_json)" in generated_html

{
    "tire_count": len(tire_specs),
    "pairing_count": default_ranking.shape[0],
    "default_winner": default_ranking.iloc[0][
        ["front_tire", "rear_tire", "total_watts"]
    ].to_dict(),
    "static_app_bytes": Path("../docs/index.html").stat().st_size,
}

{'tire_count': 11,
 'pairing_count': 99,
 'default_winner': {'front_tire': 'SL-R 28',
  'rear_tire': 'SL-R 28',
  'total_watts': 24.655593811332704},
 'static_app_bytes': 26922}

## Next Steps

The notebook and the generated Pages artifact are ready to publish. Medium
confidence pairings should be treated as a short list for real-world testing
because public wind-tunnel coverage is incomplete.